In [0]:
print("Hello fellaws")

In [0]:
%sql

CREATE OR REPLACE VIEW 03_gold.gold.kpi_total_revenue AS
SELECT ROUND(SUM(line_revenue_usd), 2) AS kpi_total_revenue_usd
FROM 03_gold.gold.fact_invoice_line_item
WHERE quantity <> 0;

In [0]:
%sql
CREATE OR REPLACE VIEW 03_gold.gold.kpi_total_quantity AS
SELECT SUM(quantity) AS kpi_total_quantity_sold
FROM 03_gold.gold.fact_invoice_line_item;

In [0]:
%sql
CREATE OR REPLACE VIEW 03_gold.gold.kpi_avg_invoice_value AS
SELECT ROUND(AVG(invoice_total_usd), 2) AS kpi_avg_invoice_value_usd
FROM (
    SELECT invoice_id, SUM(line_revenue_usd) AS invoice_total_usd
    FROM 03_gold.gold.fact_invoice_line_item
    GROUP BY invoice_id
) t;

3rd kpi

In [0]:
%sql
CREATE OR REPLACE VIEW 03_gold.gold.kpi_top_customers AS
SELECT
    f.customer_id,
    c.customer_name,
    ROUND(SUM(f.line_revenue_usd), 2) AS total_revenue_usd
FROM 03_gold.gold.fact_invoice_line_item f
LEFT JOIN 03_gold.gold.dim_customer c ON f.customer_id = c.customer_id
GROUP BY f.customer_id, c.customer_name;

In [0]:
%sql
CREATE OR REPLACE VIEW 03_gold.gold.kpi_top_products AS
SELECT
    f.product_id,
    p.product_name,
    p.category,
    ROUND(SUM(f.line_revenue_usd), 2) AS total_revenue_usd
FROM 03_gold.gold.fact_invoice_line_item f
LEFT JOIN 03_gold.gold.dim_products p ON f.product_id = p.product_id
GROUP BY f.product_id, p.product_name, p.category;

In [0]:
%sql
CREATE OR REPLACE VIEW 03_gold.gold.kpi_cancellation_rate AS
SELECT
    COUNT(*)                                                          AS total_invoices,
    SUM(CASE WHEN invoice_status = 'Cancelled' THEN 1 ELSE 0 END)    AS cancelled_invoices,
    ROUND(
        SUM(CASE WHEN invoice_status = 'Cancelled' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2
    )                                                                 AS cancellation_rate_pct
FROM 03_gold.gold.fact_invoice_line_item;

In [0]:
%sql

CREATE OR REPLACE VIEW 03_gold.gold.kpi_discount_impact AS
SELECT
    ROUND(SUM(discount * quantity * rate_to_usd), 2)  AS kpi_total_discount_given_usd,
    ROUND(AVG(discount * rate_to_usd), 2)             AS kpi_avg_discount_per_line_usd
FROM 03_gold.gold.fact_invoice_line_item;

In [0]:
%sql
CREATE OR REPLACE VIEW 03_gold.gold.kpi_revenue_by_region AS
SELECT
    region,
    ROUND(SUM(line_revenue_usd), 2) AS total_revenue_usd
FROM 03_gold.gold.fact_invoice_line_item
GROUP BY region;

In [0]:
%sql
CREATE OR REPLACE VIEW 03_gold.gold.kpi_invoices_per_customer AS
SELECT
    f.customer_id,
    c.customer_name,
    COUNT(DISTINCT f.invoice_id) AS invoice_count
FROM 03_gold.gold.fact_invoice_line_item f
LEFT JOIN 03_gold.gold.dim_customer c ON f.customer_id = c.customer_id
GROUP BY f.customer_id, c.customer_name;